In [ ]:
import scanpy as sc
import os, random
import bin2cell as b2c
import numpy as np
from matplotlib import rcParams,font_manager
import matplotlib.pyplot as plt
import pandas as pd
import re
import seaborn as sns
import anndata as ad

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

seed_value = 123

np.random.seed(seed_value)
random.seed(seed_value)

sns.set_style("white")

## 8 micron bins

In [ ]:
sample_counts_dirs = dict()
sample_images_dirs = dict()

sr_paths = ["/fs/ess/PAS2713/Data/Spatial/VisiumHD/Visium_HD_FF_Pilot/outs/spaceranger_count/dec_2025_topped",
            "/fs/ess/PAS2713/Data/Spatial/VisiumHD/Visium_HD_FFPE_Pilot/outs/spaceranger_merged_v2",
            "/fs/ess/PAS2713/Data/Spatial/VisiumHD/Visium_HD_FF_Batch1/outs/spaceranger_counts"]
bin_resolution = "square_008um"
for sr_path in sr_paths:
    samples = next(os.walk(sr_path))[1]
    samples_dict = dict(zip(samples, [sr_path] * len(samples)))
    sample_counts_dirs.update({k: os.path.join(v,k,f"outs/binned_outputs/{bin_resolution}") for k,v in samples_dict.items()})
    sample_images_dirs.update({k: os.path.join(v,k,"outs/spatial/cytassist_image.tiff") for k,v in samples_dict.items()})

In [ ]:
obs_dfs = []
for sample_id, counts_path in sample_counts_dirs.items():
    image_path = sample_images_dirs[sample_id]
    scdata_tmp = b2c.read_visium(counts_path, source_image_path = image_path)
    scdata_tmp.var_names_make_unique()
    sc.pp.filter_cells(scdata_tmp, min_counts = 1)
    sc.pp.filter_cells(scdata_tmp, min_genes = 1)
    scdata_tmp.raw = scdata_tmp
    # mitochondrial genes
    scdata_tmp.var["mt"] = scdata_tmp.var_names.str.startswith("MT-")
    # ribosomal genes
    scdata_tmp.var["ribo"] = scdata_tmp.var_names.str.startswith(("RPS", "RPL"))
    sc.pp.calculate_qc_metrics(
        scdata_tmp, qc_vars=["mt", "ribo"], inplace=True, log1p=True
    )
    scdata_tmp.obs['sample_id'] = sample_id
    obs_dfs.append(scdata_tmp.obs)
merged_obs_8um = pd.concat(obs_dfs, axis=0)
merged_obs_8um['source'] = '8 um'

## 2 micron bins

In [ ]:
sample_counts_dirs = dict()
sample_images_dirs = dict()

bin_resolution = "square_002um"
for sr_path in sr_paths:
    samples = next(os.walk(sr_path))[1]
    samples_dict = dict(zip(samples, [sr_path] * len(samples)))
    sample_counts_dirs.update({k: os.path.join(v,k,f"outs/binned_outputs/{bin_resolution}") for k,v in samples_dict.items()})
    sample_images_dirs.update({k: os.path.join(v,k,"outs/spatial/cytassist_image.tiff") for k,v in samples_dict.items()})

In [ ]:
obs_dfs = []
for sample_id, counts_path in sample_counts_dirs.items():
    image_path = sample_images_dirs[sample_id]
    scdata_tmp = b2c.read_visium(counts_path, source_image_path = image_path)
    scdata_tmp.var_names_make_unique()
    sc.pp.filter_cells(scdata_tmp, min_counts = 1)
    sc.pp.filter_cells(scdata_tmp, min_genes = 1)
    scdata_tmp.raw = scdata_tmp
    # mitochondrial genes
    scdata_tmp.var["mt"] = scdata_tmp.var_names.str.startswith("MT-")
    # ribosomal genes
    scdata_tmp.var["ribo"] = scdata_tmp.var_names.str.startswith(("RPS", "RPL"))
    sc.pp.calculate_qc_metrics(
        scdata_tmp, qc_vars=["mt", "ribo"], inplace=True, log1p=True
    )
    scdata_tmp.obs['sample_id'] = sample_id
    obs_dfs.append(scdata_tmp.obs)
merged_obs_2um = pd.concat(obs_dfs, axis=0)
merged_obs_2um['source'] = '2 um'

## 16 micron bins

In [ ]:
sample_counts_dirs = dict()
sample_images_dirs = dict()

bin_resolution = "square_016um"
for sr_path in sr_paths:
    samples = next(os.walk(sr_path))[1]
    samples_dict = dict(zip(samples, [sr_path] * len(samples)))
    sample_counts_dirs.update({k: os.path.join(v,k,f"outs/binned_outputs/{bin_resolution}") for k,v in samples_dict.items()})
    sample_images_dirs.update({k: os.path.join(v,k,"outs/spatial/cytassist_image.tiff") for k,v in samples_dict.items()})

In [ ]:
obs_dfs = []
for sample_id, counts_path in sample_counts_dirs.items():
    image_path = sample_images_dirs[sample_id]
    scdata_tmp = b2c.read_visium(counts_path, source_image_path = image_path)
    scdata_tmp.var_names_make_unique()
    sc.pp.filter_cells(scdata_tmp, min_counts = 1)
    sc.pp.filter_cells(scdata_tmp, min_genes = 1)
    scdata_tmp.raw = scdata_tmp
    # mitochondrial genes
    scdata_tmp.var["mt"] = scdata_tmp.var_names.str.startswith("MT-")
    # ribosomal genes
    scdata_tmp.var["ribo"] = scdata_tmp.var_names.str.startswith(("RPS", "RPL"))
    sc.pp.calculate_qc_metrics(
        scdata_tmp, qc_vars=["mt", "ribo"], inplace=True, log1p=True
    )
    scdata_tmp.obs['sample_id'] = sample_id
    obs_dfs.append(scdata_tmp.obs)
merged_obs_16um = pd.concat(obs_dfs, axis=0)
merged_obs_16um['source'] = '16 um'

In [ ]:
merged_obs_2um['barcode'] = merged_obs_2um.index
merged_obs_2um.groupby('sample_id').agg(
        total_bins = ('barcode', 'size'),  
        mean_UMI = ('total_counts', 'mean'),
        mean_genes = ('n_genes_by_counts', 'mean'),
        mean_mt = ('pct_counts_mt', 'mean')
    ).reset_index()

In [ ]:
merged_obs_8um['barcode'] = merged_obs_8um.index
merged_obs_8um.groupby('sample_id').agg(
        total_bins = ('barcode', 'size'),  
        mean_UMI = ('total_counts', 'mean'),
        mean_genes = ('n_genes_by_counts', 'mean'),
        mean_mt = ('pct_counts_mt', 'mean')
    ).reset_index()

In [ ]:
merged_obs_16um['barcode'] = merged_obs_16um.index
merged_obs_16um.groupby('sample_id').agg(
        total_bins = ('barcode', 'size'),  
        mean_UMI = ('total_counts', 'mean'),
        mean_genes = ('n_genes_by_counts', 'mean'),
        mean_mt = ('pct_counts_mt', 'mean')
    ).reset_index()

In [ ]:
merged_obs_all = pd.concat([merged_obs_2um, merged_obs_8um, merged_obs_16um], axis=0)
merged_obs_all.to_parquet("initial_qc_bins.parquet")

In [ ]:
merged_obs_all[["sample_id"]].value_counts()

In [ ]:
merged_obs = merged_obs_8um.copy()
merged_obs.shape

In [ ]:
merged_obs = merged_obs[merged_obs.n_genes_by_counts>0]
merged_obs.shape

In [ ]:
merged_obs.value_counts('sample_id')

In [ ]:
category_order = np.sort(merged_obs.sample_id.unique()).tolist()
merged_obs['sample_id'] = pd.Categorical(merged_obs['sample_id'], categories=category_order, ordered=True)

In [ ]:
sc.set_figure_params(dpi=200,fontsize=10, figsize=(10, 6))
sns.violinplot(y="log1p_n_genes_by_counts", hue="sample_id", data=merged_obs)
plt.show()

In [ ]:
sc.set_figure_params(dpi=200,fontsize=10, figsize=(10, 6))
sns.violinplot(y="log1p_total_counts", hue="sample_id", data=merged_obs)
plt.show()

In [ ]:
sc.set_figure_params(dpi=200,fontsize=10, figsize=(10, 6))
sns.violinplot(y="pct_counts_mt", hue="sample_id", data=merged_obs)
plt.show()